<h1>Comparing the four EEPN compensation cases: TGN model vs. full-system simulation</h1>
This notebook generalizes <code>Temporal_Gaussian_Noise_Model_EEPN_Python.ipynb</code> (which covers only carrier phase recovery, CPR) to the four receiver structures discussed in [2], classified by the order of frequency-dependent phase error (FDPE) they compensate:
<ul>
  <li><b>No compensation (LO-PC)</b> -- removes only the instantaneous LO phase itself (order -).</li>
  <li><b>CPR</b> -- compensates a constant phase offset a<sup>(0)</sup> (order 0).</li>
  <li><b>Timing recovery</b> -- additionally compensates a linear phase term a<sup>(1)</sup>*f (order 1).</li>
  <li><b>Adaptive filtering (higher-order compensation)</b> -- compensates a full degree-Ntilde polynomial, sum_n a<sup>(n)</sup>*f^n (order Ntilde).</li>
</ul>
For each case, we compare the Temporal Gaussian Noise (TGN) model's predicted time-varying EEPN distortion power against a full-system simulation using the <b>ideal, genie-aided</b> ("phase_based") realization of that compensation order -- i.e. the compensation directly uses the true LO phase <code>Rx_phi</code> rather than a receiver-realistic (data-aided or blind) estimate of it. This isolates the TGN model's accuracy from estimator noise and gives a clean, consistent comparison across all four cases; see [2] for practical data-aided/blind estimators (not included here).

[1] B. Geiger, F. Buchali, V. Aref, and L. Schmalen, "A Temporal Gaussian Noise Model for Equalization-enhanced Phase Noise," Proc. ECOC, Copenhagen, Denmark, 2025.

[2] B. Geiger, F. Buchali, V. Aref, and L. Schmalen, "Modeling and Mitigation of Equalization-Enhanced Phase Noise," Proc. ECOC, 2026.

In [ ]:
# Imports
%matplotlib notebook
import types
import numpy as np
import matplotlib.pyplot as plt
from helper_functions import ecdf
from init_parameter import init_parameter
from generate_received_symbols import generate_received_symbols
from estimate_moving_error_power import estimate_moving_error_power
from LO_phase_cancellation import LO_phase_cancellation
from CPR import CPR
from timing_recovery import timing_recovery
from full_compensation import full_compensation
from calculate_EEPN_distortion_power_LO_phase_cancellation import calculate_EEPN_distortion_power_LO_phase_cancellation
from calculate_EEPN_distortion_power_CPR import calculate_EEPN_distortion_power_CPR
from calculate_EEPN_distortion_power_timing_recovery import calculate_EEPN_distortion_power_timing_recovery
from calculate_EEPN_distortion_power_full_compensation import calculate_EEPN_distortion_power_full_compensation

### Parameters (ECOC26 reference case)

In [ ]:
cfg = types.SimpleNamespace()

cfg.symbol_rate = 130e9                 # Symbol rate in Baud
cfg.linewidth = 115e3                   # LO linewidth in Hertz
cfg.fiber_length = 2850                 # Fiber length in km

cfg.mod_order = 16                      # Modulation format
cfg.oversampling_factor = 2             # Oversampling factor

cfg.RRC_roll_off = 0.01                 # RRC roll-off factor (pulse shaping & matched filter)
cfg.RRC_span = 200                      # RRC span (pulse shaping & matched filter)

cfg.snr = 1300                          # Signal-to-noise ratio in dB (effectively noise-free, isolates EEPN)

# Kept moderate here for a fast sanity check (a few seconds); raise this
# (e.g. to 1e6 or more) for CCDF tails deep enough to be paper-quality.
cfg.num_transmission_symbols = 200000

cfg.D_cd = 23e-12/(1e-9)                # Chromatic dispersion parameter in s/(km*m)

cfg.block_length_SNR_evaluation = 1000  # Number of symbols used to estimate the SNR after transmission

# Compensation filter settings, shared by timing_recovery and full_compensation.
cfg.compensation_order = 10             # Polynomial order of the adaptive filter (Ntilde_AF in [2])
cfg.compensation_filter_taps = 60
cfg.compensation_block_length = 1000
cfg.compensation_block_overlap = 0.95

# All four cases use the ideal, genie-aided ("phase_based") compensation as
# the reference simulation; see the header cell above.
np.random.seed(1)

cfg = init_parameter(cfg)
cfg.discard_symbols_analysis = 2*cfg.CD_memory

### Transmitter, channel and receiver front end
Simulates the full pipeline once: Tx symbols -> pulse shaping -> CD channel + AWGN -> LO phase noise -> CD compensation -> matched filter -> downsampling. All four compensation cases below operate on the same <code>Rx_symbols</code>/<code>Rx_phi</code> realization.

In [ ]:
Rx_symbols, Tx_symbols, Rx_phi, Rx_symbols_up = generate_received_symbols(cfg)

### Simulation: apply the four compensation cases (ideal, genie-aided reference)

In [ ]:
Rx_symbols_LO_PC = LO_phase_cancellation(Rx_symbols, Rx_phi, cfg)
Rx_symbols_CPR, _ = CPR(Rx_symbols, Rx_phi, cfg)
Rx_symbols_TR, _, _ = timing_recovery(Rx_symbols, Rx_phi, cfg)
Rx_symbols_full, _, _ = full_compensation(Rx_symbols, Rx_phi, cfg)

### Temporal Gaussian Noise model: instantaneous EEPN distortion power
Each case's model distortion power is computed the same way, via its own dedicated <code>calculate_EEPN_distortion_power_*</code> function, independently of the compensation functions above (even though <code>timing_recovery</code> and <code>full_compensation</code> internally use the exact same windowed-fit engine).

In [ ]:
EEPN_power_model_LO_PC = calculate_EEPN_distortion_power_LO_phase_cancellation(Rx_phi, cfg)
EEPN_power_model_CPR = calculate_EEPN_distortion_power_CPR(Rx_phi, cfg)
EEPN_power_model_TR, _ = calculate_EEPN_distortion_power_timing_recovery(Rx_phi, cfg)
EEPN_power_model_full, _ = calculate_EEPN_distortion_power_full_compensation(Rx_phi, cfg)

### Evaluation: simulated error power (Sim) vs. model (system noise + EEPN power)

In [ ]:
sigma_simulation_LO_PC = estimate_moving_error_power(Rx_symbols_LO_PC, Tx_symbols, cfg)
sigma_simulation_CPR   = estimate_moving_error_power(Rx_symbols_CPR,   Tx_symbols, cfg)
sigma_simulation_TR    = estimate_moving_error_power(Rx_symbols_TR,    Tx_symbols, cfg)
sigma_simulation_full  = estimate_moving_error_power(Rx_symbols_full,  Tx_symbols, cfg)

# Match estimate_moving_error_power's own trimming convention exactly
# (discard cfg.discard_symbols_analysis symbols at the start, one more than
# that at the end) so the model curves align with the Sim curves.
D = cfg.discard_symbols_analysis
def discard(x):
    return x[D:len(x)-D-1]

sigma_model_LO_PC = cfg.system_noise_power + discard(EEPN_power_model_LO_PC)
sigma_model_CPR   = cfg.system_noise_power + discard(EEPN_power_model_CPR)
sigma_model_TR    = cfg.system_noise_power + discard(EEPN_power_model_TR)
sigma_model_full  = cfg.system_noise_power + discard(EEPN_power_model_full)

### Sanity check
Compensating more FDPE terms can only reduce the residual distortion power: mean(LO-PC) &gt;= mean(CPR) &gt;= mean(timing recovery) &gt;= mean(full compensation).

In [ ]:
mean_LO_PC = np.mean(sigma_model_LO_PC)
mean_CPR   = np.mean(sigma_model_CPR)
mean_TR    = np.mean(sigma_model_TR)
mean_full  = np.mean(sigma_model_full)

print(f"Mean distortion+noise power: LO-PC={mean_LO_PC:.3e}, CPR={mean_CPR:.3e}, "
      f"timing recovery={mean_TR:.3e}, full compensation={mean_full:.3e}")

assert mean_LO_PC >= mean_CPR >= mean_TR >= mean_full, (
    "Expected mean(LO-PC) >= mean(CPR) >= mean(timing recovery) >= mean(full compensation).")

### Comparison: distortion + noise power over time

In [ ]:
case_names = ["No compensation (LO-PC)", "CPR", "Timing recovery", "Full compensation"]
sigma_simulation_all = [sigma_simulation_LO_PC, sigma_simulation_CPR, sigma_simulation_TR, sigma_simulation_full]
sigma_model_all = [sigma_model_LO_PC, sigma_model_CPR, sigma_model_TR, sigma_model_full]

fig, axes = plt.subplots(2, 2, figsize=(10, 7))
for idx, ax in enumerate(axes.flat):
    step = cfg.block_length_SNR_evaluation
    ax.plot(sigma_simulation_all[idx][::step], label="Full system simulation")
    ax.plot(sigma_model_all[idx][::step], "--", label="Temporal GN model")
    ax.set_title(case_names[idx])
    ax.set_xlabel("Block index"); ax.set_ylabel("Noise and distortion power")
axes.flat[0].legend()
fig.suptitle("Distortion + noise power over time")
fig.tight_layout()

### Comparison: CCDF

In [ ]:
fig, axes = plt.subplots(2, 2, figsize=(10, 7))
for idx, ax in enumerate(axes.flat):
    f_sim, t_sim = ecdf(sigma_simulation_all[idx])
    f_model, t_model = ecdf(sigma_model_all[idx])
    ax.semilogy(t_sim, 1 - f_sim, linewidth=2, label="Full system simulation")
    ax.semilogy(t_model, 1 - f_model, "--", linewidth=2, label="Temporal GN model")
    ax.set_title(case_names[idx])
    ax.set_xlabel("Noise and distortion power"); ax.set_ylabel("CCDF"); ax.set_ylim([1e-3, 1])
axes.flat[0].legend()
fig.suptitle("Statistical analysis (CCDF)")
fig.tight_layout()

##### Final comments
As in <code>Temporal_Gaussian_Noise_Model_EEPN_Python.ipynb</code>, note that this idealized (genie-aided/phase_based) comparison differs from the full experimental results in [1],[2], which also include practical (data-aided/blind) DSP, dual-pol effects, etc.

If you should have any questions, comments, or remarks, please contact benedikt.geiger@kit.edu.

Written by Benedikt Geiger.